In [1]:
!pip install scikit-learn
!pip install matplotlib
!pip install mediapipe


DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.

In [5]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {
        "language": "markdown"
      },
      "source": [
        "## Dataset Structure Diagnostic",
        "This cell prints a summary of your `MP_Data` directory so you can see which actions and sequences are present."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "import os\n",
        "DATA_PATH = os.path.join('MP_Data')\n",
        "actions = ['hello', 'thanks', 'iloveyou']  # Update if needed\n",
        "if not os.path.exists(DATA_PATH):\n",
        "    print(f\"DATA_PATH '{DATA_PATH}' does not exist.\")\n",
        "else:\n",
        "    for action in actions:\n",
        "        action_path = os.path.join(DATA_PATH, action)\n",
        "        if not os.path.isdir(action_path):\n",
        "            print(f\"Action '{action}': MISSING folder!\")\n",
        "            continue\n",
        "        seqs = [d for d in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, d))]\n",
        "        print(f\"Action '{action}': {len(seqs)} sequences found.\")\n",
        "        for seq in sorted(seqs):\n",
        "            seq_folder = os.path.join(action_path, seq)\n",
        "            npy_files = [f for f in os.listdir(seq_folder) if f.endswith('.npy')]\n",
        "            print(f\"  Sequence {seq}: {len(npy_files)} .npy files\")\n"
      ]
    }
  ]
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {'language': 'markdown'},
   'source': ['## Dataset Structure Diagnostic',
    'This cell prints a summary of your `MP_Data` directory so you can see which actions and sequences are present.']},
  {'cell_type': 'code',
   'metadata': {'language': 'python'},
   'source': ['import os\n',
    "DATA_PATH = os.path.join('MP_Data')\n",
    "actions = ['hello', 'thanks', 'iloveyou']  # Update if needed\n",
    'if not os.path.exists(DATA_PATH):\n',
    '    print(f"DATA_PATH \'{DATA_PATH}\' does not exist.")\n',
    'else:\n',
    '    for action in actions:\n',
    '        action_path = os.path.join(DATA_PATH, action)\n',
    '        if not os.path.isdir(action_path):\n',
    '            print(f"Action \'{action}\': MISSING folder!")\n',
    '            continue\n',
    '        seqs = [d for d in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, d))]\n',
    '        print(f"Action \'{action}\': {len(seqs)} sequences f

In [1]:
# Data loading and checks (safer)
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

# Path for exported data, numpy arrays
DATA_PATH = os.path.join('MP_Data')

# Actions that we try to detect
actions = np.array(['cat', 'food', 'help','hello', 'thanks', 'iloveyou'])

# Thirty videos worth of data
no_sequences = 30

# Videos are going to be 30 frames in length
sequence_length = 30

# Folder start
start_folder = 30

label_map = {label:num for num, label in enumerate(actions)}

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

# --- Defensive checks before building sequences/labels ---
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"DATA_PATH '{DATA_PATH}' not found. Run the data-collection cells to create '{DATA_PATH}' with subfolders for each action.")

# Verify action folders exist and collect counts
missing_actions = [a for a in actions if not os.path.isdir(os.path.join(DATA_PATH, a))]
if missing_actions:
    raise FileNotFoundError(f"Missing action folder(s) in '{DATA_PATH}': {missing_actions}. Run data collection for these actions.")

# Build sequences and labels safely
sequences, labels = [], []
for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    # only include directories
    seq_dirs = [d for d in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, d))]
    if not seq_dirs:
        print(f"Warning: no sequence directories found for action '{action}' in {action_path}")
        continue
    # try to convert to ints (sequence folders named as integers). Skip non-numeric folder names.
    seq_ints = []
    for d in seq_dirs:
        try:
            seq_ints.append(int(d))
        except ValueError:
            print(f"Skipping non-numeric sequence folder: {d} in {action_path}")
    seq_ints = sorted(seq_ints)

    for sequence in seq_ints:
        window = []
        seq_folder = os.path.join(action_path, str(sequence))
        for frame_num in range(sequence_length):
            npy_file = os.path.join(seq_folder, f"{frame_num}.npy")
            if not os.path.exists(npy_file):
                raise FileNotFoundError(f"Expected keypoint file missing: {npy_file}. Did data collection complete successfully for this sequence?")
            res = np.load(npy_file)
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

if len(sequences) == 0:
    raise ValueError(f"No data found in '{DATA_PATH}' for actions: {list(actions)}. Please run the data collection cells and ensure .npy files exist in MP_Data/<action>/<seq>/*.npy")

# Convert to arrays
X = np.array(sequences)
y = to_categorical(labels).astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

# Model definition
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback])

# Save artifacts
model.save_weights('./model_weights.h5')
model.save('my_model')


Epoch 1/2000


/opt/homebrew/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - categorical_accuracy: 0.1695 - loss: 2.1503
Epoch 2/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - categorical_accuracy: 0.1695 - loss: 2.1503
Epoch 2/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.1535 - loss: 1.8928
Epoch 3/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.1535 - loss: 1.8928
Epoch 3/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.0869 - loss: 1.7940
Epoch 4/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.0869 - loss: 1.7940
Epoch 4/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.2214 - loss: 2.1904
Epoch 5/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.2214 - loss: 2.1904
Epoch 5/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.1618 - loss: 1.7343
Epoch 6/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.1618 - loss: 1.7343
Epoch 6/2000
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16

ValueError: The filename must end in `.weights.h5`. Received: filepath=./model_weights.h5